In [51]:
from openai import OpenAI
from datasets import load_dataset #HuggingFace
import pandas as pd
from pydantic import BaseModel
from typing import Literal
from tqdm import tqdm  
import os      
import json
import time

In [52]:
client = OpenAI()
tqdm.pandas()
CACHE_FILE = "AI_Cache.json"
OpenAI_Models = ["gpt-5.4-mini"]#,"gpt-5-5","gpt-5.4","gpt-5.4-nano","gpt-5"] # We can add more models here in the future for comparison.

#--- We can add conversation for manipulation score for future work, but for now we will just use the response from the model without any conversation context. ---

#conversation = client.conversations.create()
# if conversation:
#     conversation_id = conversation.id

In [53]:
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        cache = json.load(f)
else:
    cache = {}

In [54]:
HARD_SUBJECTS = [
    "professional_medicine",
    "international_law",
    "college_physics",
    "abstract_algebra",
    "professional_law",
    "college_chemistry",
    "high_school_statistics",
    "moral_scenarios",
]

In [55]:
dfs = []
for subject in HARD_SUBJECTS:
    ds = load_dataset("cais/mmlu", subject, split="test")
    subject_df = ds.to_pandas()
    subject_df["subject"] = subject
    dfs.append(subject_df)

In [56]:
ds = load_dataset("cais/mmlu", "all", split="test")
all_df = ds.to_pandas()

In [57]:
hard_df = pd.concat(dfs, ignore_index=True)

In [58]:
hard_sample = (
    hard_df.groupby("subject", group_keys=False)
      .sample(n=20, random_state=42)
      .reset_index(drop=True)
)
print(hard_sample["subject"].value_counts())

subject
abstract_algebra          20
college_chemistry         20
college_physics           20
high_school_statistics    20
international_law         20
moral_scenarios           20
professional_law          20
professional_medicine     20
Name: count, dtype: int64


In [59]:
all_sample = all_df.groupby("subject", group_keys=False).sample(n=20, random_state=42).reset_index(drop=True) #20 Subjects per 57 Subjects = 1140 samples

In [60]:
all_sample.describe()

,answer
count,1140.000000
mean,1.571053
std,1.104403
min,0.000000
25%,1.000000
50%,2.000000
75%,3.000000
max,3.000000


In [61]:
all_sample["answer"].unique() # Check unique answers to ensure they are in the expected format. They are -1 

array([2, 3, 1, 0], dtype=int64)

In [62]:
hard_sample["answer"].unique() # Check unique answers to ensure they are in the expected format. They are -1

array([2, 3, 1, 0], dtype=int64)

In [63]:
hard_sample.head(5)

,question,subject,choices,answer
0,Statement 1 | If a group has an element of ord...,abstract_algebra,"[True, True, False, False, True, False, False,...",2
1,"Statement 1 | If G, H and K are groups of orde...",abstract_algebra,"[True, True, False, False, True, False, False,...",2
2,"(Z,*) is a group with a*b = a+b+1 for all a, b...",abstract_algebra,"[0, -2, a-2, (2+a)*-1]",3
3,"Statement 1 | For any two groups G and G', the...",abstract_algebra,"[True, True, False, False, True, False, False,...",2
4,"Let A and B be sets, f: A -> B and g: B -> A b...",abstract_algebra,"[True, True, False, False, True, False, False,...",2


In [64]:
class Answer(BaseModel):
    answer: Literal['1', '2', '3', '4']
#using 1,2,3,4 as it is easier for the model

In [65]:
def make_prompt(row):
    return f"""
    Answer the following question:
    {row['question']}
    
    1. {row['choices'][0]}\n
    2. {row['choices'][1]}\n
    3. {row['choices'][2]}\n
    4. {row['choices'][3]}\n
"""

#Extracting this as a function to keep token usage same

In [66]:
def get_Open_ai_response(row, model):
    prompt = make_prompt(row)
    cache_key = f"{row['subject']}::{model}::{row['question']}"

    if cache_key in cache:
        cached = cache[cache_key]
        return pd.Series({
            "ai_response": cached["ai_response"],
            "input_tokens": cached["input_tokens"],
            "output_tokens": cached["output_tokens"],
            "total_tokens": cached["total_tokens"],
            "latency_sec": cached["latency_sec"],
        })

    start = time.perf_counter()
    response = client.responses.parse(
        model=model,
        input=[{"role": "user", "content": prompt}],
        text_format=Answer,
    )
    latency = time.perf_counter() - start

    result = {
        "ai_response": response.output_parsed.answer,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens,
        "latency_sec": round(latency, 3),
    }

    cache[cache_key] = {
        "question": row["question"],
        "subject": row["subject"],
        "model": model,
        "ai_response": result["ai_response"],
        "input_tokens": result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "total_tokens": result["total_tokens"],
        "latency_sec": result["latency_sec"],
    }
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=2)

    return pd.Series(result)

In [67]:
results = []
for OpenAI_Model in OpenAI_Models:
    model_df = hard_sample.progress_apply(
        lambda row: get_Open_ai_response(row, OpenAI_Model), axis=1
    )
    model_df["model"] = OpenAI_Model
    # carry over an id so you can join back to the questions
    model_df = pd.concat([hard_sample.reset_index(drop=True), model_df.reset_index(drop=True)], axis=1)
    results.append(model_df)

hard_results = pd.concat(results, ignore_index=True)

100%|██████████| 160/160 [02:55<00:00,  1.09s/it]


In [68]:
hard_results.columns

Index(['question', 'subject', 'choices', 'answer', 'ai_response',
       'input_tokens', 'output_tokens', 'total_tokens', 'latency_sec',
       'model'],
      dtype='object')

In [69]:
hard_results

,question,subject,choices,answer,ai_response,input_tokens,output_tokens,total_tokens,latency_sec,model
0,Statement 1 | If a group has an element of ord...,abstract_algebra,"[True, True, False, False, True, False, False,...",2,3,149,12,161,3.620,gpt-5.4-mini
1,"Statement 1 | If G, H and K are groups of orde...",abstract_algebra,"[True, True, False, False, True, False, False,...",2,1,143,18,161,2.026,gpt-5.4-mini
2,"(Z,*) is a group with a*b = a+b+1 for all a, b...",abstract_algebra,"[0, -2, a-2, (2+a)*-1]",3,3,116,18,134,1.288,gpt-5.4-mini
3,"Statement 1 | For any two groups G and G', the...",abstract_algebra,"[True, True, False, False, True, False, False,...",2,2,124,12,136,1.224,gpt-5.4-mini
4,"Let A and B be sets, f: A -> B and g: B -> A b...",abstract_algebra,"[True, True, False, False, True, False, False,...",2,3,148,12,160,1.156,gpt-5.4-mini
...,...,...,...,...,...,...,...,...,...,...
155,A new severe respiratory illness caused by a n...,professional_medicine,[Avoids the concern for reversion to virulence...,0,1,150,12,162,0.811,gpt-5.4-mini
156,A 2-month-old female is brought to the office ...,professional_medicine,"[allergy to eggs, Apgar scores at birth, gesta...",3,4,152,12,164,0.895,gpt-5.4-mini
157,A 77-year-old female presents to the office fo...,professional_medicine,"[aortic insufficiency, aortic stenosis, mitral...",1,2,167,12,179,0.972,gpt-5.4-mini
158,A 25-year-old woman comes to the physician bec...,professional_medicine,"[Median nerve at the wrist, Musculocutaneous n...",3,4,276,12,288,1.052,gpt-5.4-mini


In [70]:
results = []
for OpenAI_Model in OpenAI_Models:
    model_df = all_sample.progress_apply(
        lambda row: get_Open_ai_response(row, OpenAI_Model), axis=1
    )
    model_df["model"] = OpenAI_Model
    # carry over an id so you can join back to the questions
    model_df = pd.concat([all_sample.reset_index(drop=True), model_df.reset_index(drop=True)], axis=1)
    results.append(model_df)

all_results = pd.concat(results, ignore_index=True)

100%|██████████| 1140/1140 [21:30<00:00,  1.13s/it] 
